## 1. 環境設定 (Setup)

In [10]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity
from factor_analyzer.factor_analyzer import calculate_kmo


# Data Simulation

In [11]:
def simulate_efa_features(
    N=3000,                 # 新聞篇數（觀察值數量）
    K=4,                    # 潛在因子數
    feature_names=None,     # features 名稱
    make_share=True,        # 要不要做成「比例」型資料
    noise_sd=0.8,           # feature 雜訊強度（越大，因子結構越難）
    seed=42
):
    rng = np.random.default_rng(seed)

    if feature_names is None:
        feature_names = ["pos", "neg", "uncertainty", "risk", "policy", "esg", "eco"]
    P = len(feature_names)

    # 1) 生成潛在因子 F：每篇新聞都有一組因子值（這裡先用 i.i.d. 正態）
    F = rng.normal(0, 1, size=(N, K))

    # 2) 設定 loading matrix L (P x K)：你希望 EFA 找回的結構
    #    這裡的因子順序假設：0=Policy, 1=Risk, 2=Eco, 3=ESG
    L = np.zeros((P, K))

    # 對應 feature index
    idx = {name: i for i, name in enumerate(feature_names)}

    # Policy 因子 -> policy, uncertainty
    if "policy" in idx:      L[idx["policy"], 0] = 0.80
    if "uncertainty" in idx: L[idx["uncertainty"], 0] = 0.70

    # Risk 因子 -> risk, neg
    if "risk" in idx: L[idx["risk"], 1] = 0.85
    if "neg" in idx:  L[idx["neg"], 1]  = 0.75

    # Eco 因子 -> eco, pos（pos 也可能混一點 policy，讓它更像真實）
    if "eco" in idx: L[idx["eco"], 2] = 0.85
    if "pos" in idx: L[idx["pos"], 2] = 0.60

    # ESG 因子 -> esg
    if "esg" in idx: L[idx["esg"], 3] = 0.90

    # （可選）加入少量 cross-loading，讓資料更接近真實，不會太乾淨
    # 例如：pos 也會受 policy 些微影響
    if "pos" in idx: L[idx["pos"], 0] += 0.20

    # 3) 生成 features：X = F L^T + noise
    X = F @ L.T + rng.normal(0, noise_sd, size=(N, P))

    # 4) 讓資料更像「詞比例」：非負化 + 每列加總為 1
    if make_share:
        X = np.maximum(X, 0)  # 轉成非負
        row_sum = X.sum(axis=1, keepdims=True)
        X = X / np.clip(row_sum, 1e-8, None)

    df_X = pd.DataFrame(X, columns=feature_names)
    return df_X, L  # L 回傳只是方便你檢查（真實資料不會有）

In [12]:
# 例：產生 EFA 用資料 (Data Loading)
df_X, L_true = simulate_efa_features(
    N=3000,
    make_share=True,
    noise_sd=0.8
)

print(f"資料筆數: {len(df_X)}")
df_X.head()

資料筆數: 3000


,pos,neg,uncertainty,risk,policy,esg,eco
0,0.294851,0.000000,0.000000,0.000000,0.436159,0.268991,0.000000
1,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.000000,0.000000,0.230432,0.000000,0.000000,0.000000,0.769568
3,0.037898,0.196204,0.122564,0.180554,0.100966,0.000000,0.361813
4,0.000000,0.000000,0.453455,0.000000,0.317170,0.229374,0.000000


# 探索性因子分析 (Exploratory Factor Analysis, EFA)

主要流程：
1. **資料檢定**：檢查資料是否適合做因子分析 (Bartlett, KMO)。
2. **因子萃取**：決定最佳因子個數 (Scree Plot)。
3. **因子旋轉與解釋**：計算因子負荷量 (Factor Loadings) 以解釋因子意義。

## 1. 資料適動性檢定 (Data Suitability Tests)

*   **Bartlett’s Test of Sphericity**: 檢定變數之間是否互相獨立。H0: 變數間無相關（不適合做 EFA）。若 p-value < 0.05，則拒絕 H0，表示適合。
*   **KMO (Kaiser-Meyer-Olkin) Test**: 檢定變數間的偏相關性是否夠小。值介於 0~1，通常 > 0.6 表示適合。

In [13]:
# 1. Bartlett's Test
chi_square_value, p_value = calculate_bartlett_sphericity(df_X)
print(f"Bartlett's Test p-value: {p_value:.4e}")

# 2. KMO Test
kmo_all, kmo_model = calculate_kmo(df_X)
print(f"KMO Test Value: {kmo_model:.4f}")

if p_value < 0.05 and kmo_model > 0.6:
    print("=> 資料適合進行因子分析")
else:
    print("=> 資料可能不適合進行因子分析，請檢查變數相關性")

Bartlett's Test p-value: 0.0000e+00
KMO Test Value: 0.0617
=> 資料可能不適合進行因子分析，請檢查變數相關性


c:\Users\ownme\anaconda3\envs\ndhu_windows\Lib\site-packages\factor_analyzer\utils.py:244: UserWarning: The inverse of the variance-covariance matrix was calculated using the Moore-Penrose generalized matrix inversion, due to its determinant being at or very close to zero.
  warnings.warn(


## 2. 決定因子個數 (Factor Selection)

使用 **陡坡圖 (Scree Plot)** 與 **特徵值 (Eigenvalue) > 1** 準則來輔助判斷。

In [14]:
fa = FactorAnalyzer(n_factors=len(df_X.columns), rotation=None)
fa.fit(df_X)

# 取得特徵值
ev, v = fa.get_eigenvalues()

# 使用 Plotly 繪製 Steep Plot
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=list(range(1, df_X.shape[1]+1)),
    y=ev,
    mode='lines+markers',
    name='Eigenvalues'
))

# 加入 Eigenvalue = 1 的參考線
fig.add_hline(y=1, line_dash="dash", line_color="red", annotation_text="Eigenvalue=1")

fig.update_layout(
    title='Scree Plot (陡坡圖)',
    xaxis_title='Factors',
    yaxis_title='Eigenvalue',
    template='plotly_white'
)
fig.show()

n_factors_ev1 = sum(ev > 1)
print(f"特徵值 > 1 的因子個數: {n_factors_ev1}")
print(f"各因子特徵值: {np.round(ev, 2)}")

c:\Users\ownme\anaconda3\envs\ndhu_windows\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


特徵值 > 1 的因子個數: 3
各因子特徵值: [1.5  1.36 1.2  0.97 0.96 0.95 0.07]


## 3. 執行因子分析與視覺化 (Factor Extraction & Visualization)

*   **Rotation**: 使用 `promax` (非正交旋轉)，因為新聞語意特徵（如風險、政策）通常具有相關性。
*   **Factors**: 設定為 4 (模擬設定是 4，通常由 Scree Plot 判斷)。

In [15]:
# 設定因子數為 4，並使用 promax 旋轉
fa = FactorAnalyzer(n_factors=4, rotation='promax')
fa.fit(df_X)

# 取得因子負荷量 (Loading Matrix)
loadings = pd.DataFrame(fa.loadings_, index=df_X.columns, columns=[f'Factor{i+1}' for i in range(4)])

# 使用 Plotly 繪製 Heatmap
fig = px.imshow(
    loadings,
    x=loadings.columns,
    y=loadings.index,
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    text_auto='.2f',
    aspect='auto',
    title='Factor Loadings (Promax Rotation)'
)
fig.update_layout(
    template='plotly_white'
)
fig.show()

print("因子負荷量表:")
loadings

c:\Users\ownme\anaconda3\envs\ndhu_windows\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.



因子負荷量表:


,Factor1,Factor2,Factor3,Factor4
pos,-0.096439,-0.097885,-0.065012,0.003080
neg,-0.316183,1.281450,-0.343228,0.126377
uncertainty,-0.076958,-0.076399,-0.079265,-0.336662
risk,1.283102,-0.313716,-0.352487,0.132173
policy,-0.079633,-0.068978,-0.072997,-0.332154
esg,-0.326100,-0.313390,1.229636,0.135706
eco,-0.251747,-0.227377,-0.225166,0.839965


### 解釋變異量 (Factor Variance)
檢視每個因子解釋了多少資料變異，以及累計解釋變異量。

In [16]:
variance_df = pd.DataFrame(fa.get_factor_variance(), 
                           index=['SS Loadings', 'Proportion Var', 'Cumulative Var'],
                           columns=[f'Factor{i+1}' for i in range(4)])

variance_df

,Factor1,Factor2,Factor3,Factor4
SS Loadings,1.937605,1.910622,1.820595,0.981076
Proportion Var,0.276801,0.272946,0.260085,0.140154
Cumulative Var,0.276801,0.549747,0.809832,0.949985
